<a href="https://colab.research.google.com/github/THEJoshinator20/ST-554-Project1-Template/blob/main/Task1/Task1_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1: Gradient Descent Algorithms

This task involves writing two gradient descent type algorithms to find the optimal constant to use for squared error loss (ends up being the sample mean) and to find the optimal intercept and slope from a simple linear regression model.

## Load and clean data set

In [ ]:
# install and import ucimlrepo
!pip install ucimlrepo

import ucimlrepo as uci
import numpy as np

In [125]:
air_quality = uci.fetch_ucirepo(id=360)
# view data set
air_quality = air_quality.data.features
air_quality.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


Remove any observations where the C6H6(GT) or CO(GT) are -200 as these represent missing values

In [127]:
air_quality = air_quality[(air_quality['C6H6(GT)'] != -200) & (air_quality['CO(GT)'] != -200)]

## Prediction of C6H6(GT)

### Grid Search Algorithm

Implement a grid search to find the optimal value of c based off the data set.

First, do it using just y.

Find the first and third quartiles of C6H6(GT) to determine reasonable values for grid search

In [128]:
q1 = np.percentile(air_quality['C6H6(GT)'], 25)
q3 = np.percentile(air_quality['C6H6(GT)'], 75)

print(q1)
print(q3)

4.6
14.3


 Create a root mean squared error (objective) function

In [129]:
def rmse(y, c: float) -> float:
  """
  Calculates Root Mean Squared Error (RMSE) for a given value of c.
  """
  # calculate and return RMSE
  return np.sqrt(np.mean((y - c) ** 2))

Use a list comprehensive to loop over the grid of c values, finding the RMSE for each value of c.

In [130]:
# create grid of values to try
grid = np.linspace(q1, q3, 100)
# calculate RMSE for each value in grid
rmse_values = [rmse(air_quality['C6H6(GT)'], c) for c in grid]

Determine which value of c gives the optimal (smallest) RMSE.

In [132]:
# minimum rmse value
print(min(rmse_values))

# find value of c that gives minimum rmse
print(grid[np.argmin(rmse_values)])

7.440564471926773
10.282828282828284


Wrap it all into one function

In [133]:
def find_optimal_c(y) -> float:
  """
  Function to find the optimal value of c for a given column.
  """
  # find first and third quartile
  q1, q3 = np.percentile(y, [25, 75])

  # create grid using quartiles
  grid = np.linspace(q1, q3, 100)

  # calculate RMSE for each value in grid
  rmse_values = [rmse(air_quality['C6H6(GT)'], c) for c in grid]

  # output optimal value of c
  return grid[np.argmin(rmse_values)]

Test function on C6H6(GT)

In [147]:
c = find_optimal_c(air_quality['C6H6(GT)'])
print(f"Optimal c value: {round(c, 4)}")

Optimal c value: 10.2756


Test function on PT08.S1(CO) to make sure algorithm generalizes

In [149]:
c = find_optimal_c(air_quality['PT08.S1(CO)'])
print(f"Optimal c value: {round(c, 4)}")

Optimal c value: 946.0


Finding optimal c using y and another numeric variable: implement the grid search to find the optimal pair of values for b0 and b1 using PT08.S1(CO) as your x variable and C6H6(GT) as your y variable.

Create function to calculate RMSE

In [137]:
def rmse_xy(x, y, b0, b1):
  """
  Calculates Root Mean Squared Error (RMSE) for a given value of x, y, b0, and b1.
  """
  # calculate c
  c = b0 + b1 * x
  # return RMSE
  return np.sqrt(np.mean((y - c)**2))

Create function to find optimal v0 and b1 values

In [150]:
def find_optimal_betas(x, y):
  """
  Finds optimal values of b0 and b1 for a given x and y variable.
  """

  # create b0 and b1 values
  b0_vals, b1_vals = np.arange(-25, -15, 0.1), np.arange(-5, 5, 0.01)

  # create grid
  grid = [(b0, b1) for b0 in b0_vals for b1 in b1_vals]

  # calculate RMSE for each value of b0 and b1
  rmse_values = [rmse_xy(x, y, b0, b1) for b0, b1 in grid]

  # find best values of b0 and b1
  best_b0, best_b1 = grid[np.argmin(rmse_values)]

  return best_b0, best_b1


Test with x = PT08.S1(CO) and y = C6H6(GT)

In [152]:
b0, b1 = find_optimal_betas(air_quality['PT08.S1(CO)'], air_quality['C6H6(GT)'])
print(f"Optimal b0 = {round(b0, 4)}, optimal b1 = {round(b1, 4)}")

Optimal b0 = -23.0, optimal b1 = 0.03


Use these values to predict a new C6H6(GT) for a PT08.S1(CO) of 946, 1075, and 1246.

Predicted value for PT08.S1(CO) = 946: 5.38
<br>
Predicted value for PT08.S1(CO) = 1075: 9.25
<br>
Predicted value for PT08.S1(CO) = 1246: 14.38

In [145]:
for num in [946, 1075, 1246]:
  pred = b0 + b1 * num
  print(round(pred, 4))

5.38
9.25
14.38
